# Лабораторная работа 1. NumPy


## 1. Числа с плавающей точкой

Сумма 0.1, 0.2 и 0.3, машинное эпсилон и вычитание близких чисел.


In [1]:
import numpy as np
from decimal import Decimal

arr32 = np.array([0.1, 0.2, 0.3], dtype=np.float32)
sum32 = np.sum(arr32, dtype=np.float32)
exact = Decimal('0.1') + Decimal('0.2') + Decimal('0.3')
err = Decimal.from_float(float(sum32)) - exact

print('1.1. Сумма float32:', format(float(sum32), '.17g'))
print('     Точное десятичное значение:', exact)
print('     Абсолютная ошибка:', abs(err))

print('\n1.2. Машинное эпсилон:')
print('     float32:', np.finfo(np.float32).eps)
print('     float64:', np.finfo(np.float64).eps)

a32, b32 = np.float32(1.00000001), np.float32(1.0)
a64, b64 = np.float64(1.00000001), np.float64(1.0)
print('\n1.3. Ожидаемая разность: 0.00000001')
print('     float32:', a32 - b32)
print('     float64:', a64 - b64)
print('     float32 округляет 1.00000001 до 1.0, поэтому разность равна 0.')


1.1. Сумма float32: 0.60000002384185791
     Точное десятичное значение: 0.6
     Абсолютная ошибка: 2.384185791015625E-8

1.2. Машинное эпсилон:
     float32: 1.1920929e-07
     float64: 2.220446049250313e-16

1.3. Ожидаемая разность: 0.00000001
     float32: 0.0
     float64: 9.99999993922529e-09
     float32 округляет 1.00000001 до 1.0, поэтому разность равна 0.


В float32 сумма немного отличается от 0.6. Число 1.00000001 округляется до 1, поэтому разность получается нулевой. В float64 точности хватает, чтобы сохранить эту разницу.


## 2. Индексация и broadcasting

В срезе строки 1–3 и столбцы 2–4 взяты по индексам, включая обе границы.


In [2]:
import numpy as np

# Отрицательные числа оставляем специально для задания 2.3.
A = np.arange(-10, 10).reshape(4, 5)
print('Исходный A (4 x 5):\n', A)

print('\n2.1. Вторая строка (индекс 1):', A[1, :])
print('     Столбец с индексом 3:', A[:, 3])
print('     Строки с индексами 1,2,3 и столбцы 2,3,4:\n', A[1:4, 2:5])

# np.ix_ делает декартово произведение индексов: результат 3 x 2.
rows = [0, 2, 3]
cols = [1, 4]
selected = A[np.ix_(rows, cols)]
print('\n2.2. Fancy-индексация (3 x 2):\n', selected)

nonnegative = A.copy()
nonnegative[nonnegative < 0] = 0
print('\n2.3. Отрицательные заменены нулями:\n', nonnegative)

B = np.arange(1, 5).reshape(4, 1)
C = np.array([[10, 20, 30, 40, 50]])
D = B + C
print('\n2.4. B.shape:', B.shape, 'C.shape:', C.shape)
print('     Broadcasting: B + C, форма', D.shape, '\n', D)
assert D.shape == (4, 5)


Исходный A (4 x 5):
 [[-10  -9  -8  -7  -6]
 [ -5  -4  -3  -2  -1]
 [  0   1   2   3   4]
 [  5   6   7   8   9]]

2.1. Вторая строка (индекс 1): [-5 -4 -3 -2 -1]
     Столбец с индексом 3: [-7 -2  3  8]
     Строки с индексами 1,2,3 и столбцы 2,3,4:
 [[-3 -2 -1]
 [ 2  3  4]
 [ 7  8  9]]

2.2. Fancy-индексация (3 x 2):
 [[-9 -6]
 [ 1  4]
 [ 6  9]]

2.3. Отрицательные заменены нулями:
 [[0 0 0 0 0]
 [0 0 0 0 0]
 [0 1 2 3 4]
 [5 6 7 8 9]]

2.4. B.shape: (4, 1) C.shape: (1, 5)
     Broadcasting: B + C, форма (4, 5) 
 [[11 21 31 41 51]
 [12 22 32 42 52]
 [13 23 33 43 53]
 [14 24 34 44 54]]


`np.ix_` выбирает все сочетания заданных строк и столбцов. При сложении массивов (4, 1) и (1, 5) получается массив (4, 5).


## 3. Memory-mapped файл

Создание массива на диске и сравнение обхода по строкам и столбцам.


In [3]:
import numpy as np
from pathlib import Path

file_path = Path('numpy_memmap.dat')
rng = np.random.default_rng(42)

# 3.1. Создание массива на диске, ~8 МБ.
mm = np.memmap(file_path, dtype='float64', mode='w+', shape=(1000, 1000))
for start in range(0, 1000, 100):
    mm[start:start + 100, :] = rng.random((100, 1000))
mm.flush()
del mm

# 3.2. Открываем только отображение файла, не загружая все 8 МБ в память.
mapped = np.memmap(file_path, dtype='float64', mode='r', shape=(1000, 1000))
print('3.1. Создан файл:', file_path, '| размер:', file_path.stat().st_size, 'байт')
print('3.2. Первые 10 элементов строки 500:', mapped[500, :10])
print('     Первые 10 элементов столбца 500:', mapped[:10, 500])

def sum_by_rows(m):
    s = 0.0
    for i in range(m.shape[0]):
        s += np.sum(m[i, :])
    return s

def sum_by_columns(m):
    s = 0.0
    for j in range(m.shape[1]):
        s += np.sum(m[:, j])
    return s

assert np.isclose(sum_by_rows(mapped), sum_by_columns(mapped))
print('3.3. Суммы при обоих способах обхода совпадают (с допустимой погрешностью).')


3.1. Создан файл: numpy_memmap.dat | размер: 8000000 байт
3.2. Первые 10 элементов строки 500: [0.36055581 0.05036582 0.97409122 0.5715947  0.77677195 0.35830264
 0.38796874 0.28498029 0.76531126 0.43034847]
     Первые 10 элементов столбца 500: [0.73670569 0.67185419 0.0338778  0.41402758 0.01885158 0.17120494
 0.91484235 0.50154091 0.01828454 0.69085458]
3.3. Суммы при обоих способах обхода совпадают (с допустимой погрешностью).


### Время обхода


In [4]:
%timeit -n 3 -r 3 sum_by_rows(mapped)

4.98 ms ± 213 μs per loop (mean ± std. dev. of 3 runs, 3 loops each)


In [5]:
%timeit -n 3 -r 3 sum_by_columns(mapped)

5.77 ms ± 487 μs per loop (mean ± std. dev. of 3 runs, 3 loops each)


Массив хранится построчно, поэтому соседние элементы строки лежат рядом в памяти. При обходе столбца приходится переходить через целую строку. Это влияет на работу кэша; разница во времени зависит от компьютера.


## 4. Разреженные матрицы

Форматы CSR, COO и CSC, умножение на вектор и сравнение с плотной матрицей.


In [6]:
import numpy as np
from scipy import sparse

# 4.1. Размер 5x5, плотность 0.2 => 5 ненулевых элементов.
csr = sparse.random(5, 5, density=0.2, format='csr', random_state=42)
print('4.1. CSR-матрица:\n', csr.toarray())
print('     Ненулевых элементов:', csr.nnz, 'из 25; плотность:', csr.nnz / 25)

# 4.2. Переводим в другие форматы хранения.
coo = csr.tocoo()
csc = csr.tocsc()
print('\n4.2. Форматы:', type(csr).__name__, type(coo).__name__, type(csc).__name__)
print('     COO (строка, столбец, значение):')
for r, c, value in zip(coo.row, coo.col, coo.data):
    print(f'     ({r}, {c}) = {value:.6f}')
print('     CSC.indptr:', csc.indptr, 'CSC.indices:', csc.indices)

# 4.3. Матрично-векторное произведение.
v = np.arange(1, 6, dtype=float)
result = csr @ v
print('\n4.3. Вектор:', v)
print('     CSR @ вектор:', result)
np.testing.assert_allclose(result, csr.toarray() @ v)


4.1. CSR-матрица:
 [[0.43194502 0.         0.         0.         0.        ]
 [0.         0.         0.61185289 0.52475643 0.        ]
 [0.         0.         0.         0.         0.        ]
 [0.         0.30424224 0.         0.         0.29122914]
 [0.         0.         0.         0.         0.        ]]
     Ненулевых элементов: 5 из 25; плотность: 0.2

4.2. Форматы: csr_matrix coo_matrix csc_matrix
     COO (строка, столбец, значение):
     (0, 0) = 0.431945
     (1, 2) = 0.611853
     (1, 3) = 0.524756
     (3, 1) = 0.304242
     (3, 4) = 0.291229
     CSC.indptr: [0 1 2 3 4 5] CSC.indices: [0 3 1 1 3]

4.3. Вектор: [1. 2. 3. 4. 5.]
     CSR @ вектор: [0.43194502 3.93458441 0.         2.06463019 0.        ]


In [7]:
# 4.4. Сравнение на одинаковой матрице и одинаковом векторе.
big_sparse = sparse.random(1000, 1000, density=0.01, format='csr', random_state=123)
big_dense = big_sparse.toarray()
vector = np.random.default_rng(123).random(1000)
np.testing.assert_allclose(big_sparse @ vector, big_dense @ vector, rtol=1e-12, atol=1e-12)

print('4.4. 1000x1000, density=0.01')
print('     CSR хранение (data, indices, indptr):',
      big_sparse.data.nbytes + big_sparse.indices.nbytes + big_sparse.indptr.nbytes, 'байт')
print('     Dense хранение:', big_dense.nbytes, 'байт')
print('     Результаты умножения совпадают.')


4.4. 1000x1000, density=0.01
     CSR хранение (data, indices, indptr): 124004 байт
     Dense хранение: 8000000 байт
     Результаты умножения совпадают.


### Время умножения


In [8]:
%timeit -n 10 -r 3 big_sparse @ vector

The slowest run took 4.71 times longer than the fastest. This could mean that an intermediate result is being cached.
16.5 μs ± 12.8 μs per loop (mean ± std. dev. of 3 runs, 10 loops each)


In [9]:
%timeit -n 10 -r 3 big_dense @ vector

131 μs ± 12.3 μs per loop (mean ± std. dev. of 3 runs, 10 loops each)


При плотности 1% CSR занимает гораздо меньше памяти, поскольку хранит только ненулевые элементы и их индексы. Для сравнения времени использованы одна и та же матрица и один вектор.


## 5. Python и Cython

Сумма квадратов для списка из 10⁷ элементов. Значения повторяются от 0 до 9, поэтому сумма помещается в 64-битное целое.


In [10]:
def sum_squares_python(values):
    """Сумма квадратов элементов списка целых чисел (чистый Python)."""
    total = 0
    for x in values:
        total += x * x
    return total

print('5.1. Проверка Python-функции:', sum_squares_python([1, 2, 3]))
assert sum_squares_python([1, 2, 3]) == 14


5.1. Проверка Python-функции: 14


### Функция на Cython


In [ ]:
%%writefile sum_squares_cython.pyx
# cython: language_level=3
# Реализация для списка целых чисел.
# Важное условие: сумма квадратов должна помещаться в signed long long.

cpdef long long sum_squares_cython(list values):
    cdef Py_ssize_t i, n = len(values)
    cdef long long x, total = 0
    for i in range(n):
        x = <long long>values[i]
        total += x * x
    return total

### Сборка


In [ ]:
%%writefile setup.py
from setuptools import setup, Extension
from Cython.Build import cythonize

extensions = [Extension("sum_squares_cython", ["sum_squares_cython.pyx"])]
setup(
    name="numpy-lab-cython",
    ext_modules=cythonize(extensions, compiler_directives={"language_level": 3}),
)

In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, 'setup.py', 'build_ext', '--inplace'], check=True)


### Сравнение результатов и времени


In [14]:
import importlib
importlib.invalidate_caches()
from sum_squares_cython import sum_squares_cython

N = 10**7
# Список из 10^7 элементов, значения 0..9. Сумма квадратов помещается в int64.
values = [i % 10 for i in range(N)]
python_result = sum_squares_python(values)
cython_result = sum_squares_cython(values)
print('5.3. Размер списка:', N)
print('     Python:', python_result)
print('     Cython:', cython_result)
assert python_result == cython_result
print('     Результаты совпадают.')


5.3. Размер списка: 10000000
     Python: 285000000
     Cython: 285000000
     Результаты совпадают.


In [15]:
%timeit -n 1 -r 3 sum_squares_python(values)

244 ms ± 2.38 ms per loop (mean ± std. dev. of 3 runs, 1 loop each)


In [16]:
%timeit -n 3 -r 3 sum_squares_cython(values)

17.9 ms ± 729 μs per loop (mean ± std. dev. of 3 runs, 3 loops each)


В Cython переменные имеют типы C, и цикл выполняется в скомпилированном коде. В приведённом запуске он быстрее Python. На другом компьютере время будет отличаться.
